# SLURM from Jupyter: Student Lab

Hands-on lab for learning SLURM from a notebook using the local `slurm_magic.py` file in this folder.

What you will practice:
- load the local SLURM magics extension
- inspect the cluster with `sinfo` and `squeue`
- submit a batch job with `%%sbatch` and read its output file
- run a Python script as a SLURM job
- use job arrays for parameter sweeps
- chain jobs with `--dependency`
- monitor job status from Python

Tip: run the notebook on a SLURM-connected machine. All examples use the `plgrid-testing` partition (1-hour limit) so they start quickly — except array jobs and dependency jobs, which require the `plgrid` partition (see notes in those sections).

## 1. Load the local extension

In [1]:
from IPython import get_ipython
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path

ip = get_ipython()

# Look for slurm_magic.py next to this notebook first, then search upward.
candidate = Path.cwd() / 'slurm_magic.py'
if not candidate.exists():
    matches = list(Path.cwd().rglob('slurm_magic.py'))
    if not matches:
        raise FileNotFoundError('Could not find slurm_magic.py')
    candidate = sorted(matches)[0]

spec = spec_from_file_location('local_slurm_magic', candidate)
module = module_from_spec(spec)
spec.loader.exec_module(module)
module.load_ipython_extension(ip)
print(f'Loaded slurm_magic from {candidate}')

Loaded slurm_magic from /net/afscra/people/plgdawidz112002/slurm_magic/slurm_magic.py


## 2. Cluster snapshot

Before submitting any job, check what is available.  
`sinfo -s` gives a compact summary; the columns `NODES(A/I/O/T)` mean **Allocated / Idle / Other / Total**.

Questions to answer:
- Which partitions have idle nodes right now?
- What is the time limit of `plgrid-testing`?
- How many GPU nodes are available?

In [2]:
%slurm display raw
%sinfo -s

PARTITION       AVAIL  TIMELIMIT   NODES(A/I/O/T) NODELIST
all                up 3-00:00:00   467/295/35/797 ac[0001-0788],ag[0001-0009]
cpu                up 3-00:00:00   464/289/35/788 ac[0001-0788]
cpu-lowprio        up 3-00:00:00   464/289/35/788 ac[0001-0788]
cpu-bigmem         up 3-00:00:00     18/235/3/256 ac[0129-0384]
cpu-lowmem         up 3-00:00:00    446/54/32/532 ac[0001-0128,0385-0788]
gpu                up 3-00:00:00          3/6/0/9 ag[0001-0009]
plgrid*            up 3-00:00:00   464/289/35/788 ac[0001-0788]
plgrid-bigmem      up 3-00:00:00     18/235/3/256 ac[0129-0384]
plgrid-gpu-v100    up 2-00:00:00          3/6/0/9 ag[0001-0009]
plgrid-long        up 7-00:00:00    446/54/32/532 ac[0001-0128,0385-0788]
plgrid-now         up   12:00:00    446/54/32/532 ac[0001-0128,0385-0788]
plgrid-testing     up    1:00:00    446/54/32/532 ac[0001-0128,0385-0788]


In [3]:
# Switch to pandas mode and inspect partitions as a DataFrame.
%slurm display pandas
partitions = %sinfo -s
partitions

,PARTITION,AVAIL,TIMELIMIT,NODES(A/I/O/T),NODELIST
0,all,up,3-00:00:00,467/295/35/797,"ac[0001-0788],ag[0001-0009]"
1,cpu,up,3-00:00:00,464/289/35/788,ac[0001-0788]
2,cpu-lowprio,up,3-00:00:00,464/289/35/788,ac[0001-0788]
3,cpu-bigmem,up,3-00:00:00,18/235/3/256,ac[0129-0384]
4,cpu-lowmem,up,3-00:00:00,446/54/32/532,"ac[0001-0128,0385-0788]"
5,gpu,up,3-00:00:00,3/6/0/9,ag[0001-0009]
6,plgrid*,up,3-00:00:00,464/289/35/788,ac[0001-0788]
7,plgrid-bigmem,up,3-00:00:00,18/235/3/256,ac[0129-0384]
8,plgrid-gpu-v100,up,2-00:00:00,3/6/0/9,ag[0001-0009]
9,plgrid-long,up,7-00:00:00,446/54/32/532,"ac[0001-0128,0385-0788]"


## 3. Queue inspection

`squeue` shows all jobs in the scheduler. The `-u $USER` flag filters to your jobs only.  
Key columns:
- `ST` — state: `PD` = pending, `R` = running, `CG` = completing
- `TIME` — elapsed time in `HH:MM:SS`
- `NODELIST(REASON)` — assigned node or why the job waits

In [4]:
%slurm display pandas
queue = %squeue -u $USER
queue

,JOBID,PARTITION,NAME,USER,ST,TIME,NODES,NODELIST(REASON)


## 4. First batch job — output files

Every `%%sbatch` cell is sent as a shell script to SLURM.  
Important flags used here:
- `--output` — path for stdout (the `%j` placeholder is replaced by the job ID)
- `--error` — separate file for stderr
- `-p plgrid-testing` — short-limit partition for quick testing

After the job finishes, we will read the output file from Python.

In [14]:
%%sbatch -N1 -c1 -t 00:02:00 -p plgrid-testing --job-name=lab-hello --output=lab-hello-%j.out --error=lab-hello-%j.err
#!/bin/bash
echo "=== System info ==="
echo "Hostname:    $(hostname)"
echo "CPU cores:   $(nproc)"
echo "Memory (MB): $(free -m | awk '/^Mem/{print $2}')"
echo "Date:        $(date)"
echo "Working dir: $SLURM_SUBMIT_DIR"
echo "Job ID:      $SLURM_JOB_ID"

Submitted batch job 20367609


### 4a. Read the output file

Replace `JOBID` below with the number printed by `%%sbatch`.  
The cell polls `squeue` until the job is gone, then reads the file.

In [15]:
import time, subprocess, os

JOBID = '20367609'  # <-- paste the job ID here
out_file = f'lab-hello-{JOBID}.out'

print(f'Waiting for job {JOBID} to finish...', flush=True)
while True:
    result = subprocess.run(['squeue', '-j', JOBID, '-h'], capture_output=True, text=True)
    if not result.stdout.strip():
        break
    print('.', end='', flush=True)
    time.sleep(5)

print(f'\nJob done. Output from {out_file}:')
print(open(out_file).read() if os.path.exists(out_file) else 'File not found — check the path.')

Waiting for job 20367609 to finish...
.
Job done. Output from lab-hello-20367609.out:
=== System info ===
Hostname:    ac0788
CPU cores:   1
Memory (MB): 191910
Date:        Wed Jun 10 13:50:29 CEST 2026
Working dir: /net/afscra/people/plgdawidz112002/slurm_magic
Job ID:      20367609



## 5. Python script as a SLURM job

A common pattern: write a Python script with `%%writefile`, then submit it with `%%sbatch`.  
Here we estimate π using a Monte Carlo method — a real numerical computation.

The script accepts the number of random samples as `$1` so we can vary it from the batch script.

In [7]:
%%writefile estimate_pi.py
import random
import sys

N = int(sys.argv[1]) if len(sys.argv) > 1 else 1_000_000
inside = sum(
    1 for _ in range(N)
    if random.random() ** 2 + random.random() ** 2 < 1.0
)
pi_estimate = 4 * inside / N
print(f'N={N:,}  pi≈{pi_estimate:.6f}  error={abs(pi_estimate - 3.14159265):.6f}')

Overwriting estimate_pi.py


In [16]:
%%sbatch -N1 -c1 -t 00:02:00 -p plgrid-testing --job-name=lab-pi --output=lab-pi-%j.out
#!/bin/bash
python3 estimate_pi.py 5000000

Submitted batch job 20367610


## 6. Job arrays — parameter sweep

Job arrays let you run the same script many times with different parameters using a single submission.  
SLURM sets `$SLURM_ARRAY_TASK_ID` to each value in the range you specify.

This example runs the π estimator with three different sample sizes: 100 k, 200 k, 300 k.

> **Why not `plgrid-testing` here?**  
> That partition enforces `MaxSubmitPU=1` — only **one submitted job per user at a time**.  
> An array of 3 counts as 3 submissions, so it fails immediately with `QOSMaxSubmitJobPerUserLimit`.  
> We use the default `plgrid` partition (no such per-user limit) for array jobs.  
> You can always check QOS limits yourself:
> ```
> sacctmgr show qos format=Name,MaxSubmitJobsPerUser,MaxJobsPerUser -P
> ```

In [22]:
%%sbatch -N1 -c1 -t 00:03:00 -p plgrid --job-name=lab-pi-sweep --array=1-3 --output=lab-pi-sweep-%A_%a.out
#!/bin/bash
# SLURM_ARRAY_TASK_ID is 1..3; multiply by 100 000 000 to get the sample count.
N=$(( SLURM_ARRAY_TASK_ID * 100000000 ))
echo "Task $SLURM_ARRAY_TASK_ID: N=$N"
python3 estimate_pi.py $N

Submitted batch job 20367616


In [18]:
# Watch array tasks in the queue.  %A is the array job id, %a the task index.
%slurm display pandas
%squeue -u $USER

,JOBID,PARTITION,NAME,USER,ST,TIME,NODES,NODELIST(REASON)
0,20367611_1,plgrid,lab-pi-s,plgdawid,R,0:03,1,ac0788
1,20367611_2,plgrid,lab-pi-s,plgdawid,R,0:03,1,ac0788
2,20367611_3,plgrid,lab-pi-s,plgdawid,R,0:03,1,ac0788


In [21]:
import glob

ARRAY_JOBID = '20367611'  # <-- paste the array job ID printed above (the number before _1, _2, _3)

files = sorted(glob.glob(f'lab-pi-sweep-{ARRAY_JOBID}_*.out'))
for f in files:
    print(f'--- {f} ---')
    print(open(f).read().strip())
    print()

--- lab-pi-sweep-20367611_1.out ---
Task 1: N=100000000
N=100,000,000  pi≈3.141327  error=0.000265

--- lab-pi-sweep-20367611_2.out ---
Task 2: N=200000000
N=200,000,000  pi≈3.141743  error=0.000151

--- lab-pi-sweep-20367611_3.out ---
Task 3: N=300000000
N=300,000,000  pi≈3.141555  error=0.000038



## 7. Job dependencies — pipeline

`--dependency=afterok:JOBID` tells SLURM to start a job only after another finishes successfully.  
This is how you build multi-stage pipelines without polling.

**Important:** the dependency must be registered *before* job A finishes — SLURM removes completed
jobs from its records after a short window. The safe pattern is to submit both jobs in a single
Python cell: capture job A's ID from stdout, then immediately submit job B with that ID.

> **Partition note:** same as array jobs — `plgrid-testing` allows only 1 submitted job per user,
> so submitting A + B (even with a dependency) hits the limit. We use `plgrid` here too.

Example pipeline:
1. **Job A** — generates a list of random numbers and saves it to a JSON file.
2. **Job B** — waits for A (`afterok`), then reads the file and computes statistics.

In [20]:
import subprocess, re

COMMON = ['-N1', '-c1', '-t', '00:02:00', '-p', 'plgrid']

def sbatch(script, *extra_flags):
    """Submit a batch script, return the job ID string."""
    r = subprocess.run(['sbatch'] + COMMON + list(extra_flags),
                       input=script, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.strip())
    return re.search(r'\d+', r.stdout).group()

# --- Job A: generate data ---
script_a = """\
#!/bin/bash
python3 -c "
import json, random
data = [random.randint(1, 1000) for _ in range(200)]
json.dump(data, open('lab_dep_data.json', 'w'))
print(f'Generated {len(data)} numbers, first 10: {data[:10]}')
"
"""

job_a = sbatch(script_a,
               '--job-name=lab-dep-generate',
               '--output=lab-dep-generate-%j.out')
print(f'Job A submitted: {job_a}')

# --- Job B: stats — submitted immediately, before A can finish ---
script_b = """\
#!/bin/bash
python3 -c "
import json
data = json.load(open('lab_dep_data.json'))
print(f'Count: {len(data)}')
print(f'Sum:   {sum(data)}')
print(f'Mean:  {sum(data)/len(data):.2f}')
print(f'Min:   {min(data)}')
print(f'Max:   {max(data)}')
"
"""

job_b = sbatch(script_b,
               '--job-name=lab-dep-stats',
               '--output=lab-dep-stats-%j.out',
               f'--dependency=afterok:{job_a}')
print(f'Job B submitted: {job_b}  (will start after {job_a} succeeds)')

Job A submitted: 20367614
Job B submitted: 20367615  (will start after 20367614 succeeds)


## 8. Monitoring — status from Python

You can programmatically parse `squeue` output to build simple dashboards or wait loops.  
The cell below prints a live status table every 10 seconds until all your jobs are done.

In [23]:
import subprocess, time, os
from IPython.display import clear_output

def get_my_jobs():
    r = subprocess.run(
        ['squeue', '-u', os.environ['USER'],
         '--format=%i|%j|%T|%M|%R', '--noheader'],
        capture_output=True, text=True,
    )
    rows = []
    for line in r.stdout.strip().splitlines():
        parts = line.split('|')
        if len(parts) == 5:
            rows.append(parts)
    return rows

for tick in range(30):          # max 5 minutes
    clear_output(wait=True)
    jobs = get_my_jobs()
    if not jobs:
        print('No jobs in the queue.')
        break
    print(f'{"JOBID":>12}  {"NAME":<25}  {"STATE":<12}  {"TIME":>8}  REASON')
    print('-' * 80)
    for jobid, name, state, elapsed, reason in jobs:
        print(f'{jobid:>12}  {name:<25}  {state:<12}  {elapsed:>8}  {reason}')
    print(f'\n[tick {tick+1}] refreshing in 10 s — interrupt the cell to stop')
    time.sleep(10)

       JOBID  NAME                       STATE             TIME  REASON
--------------------------------------------------------------------------------
  20367616_2  lab-pi-sweep               RUNNING           0:23  ac0788
  20367616_3  lab-pi-sweep               RUNNING           0:23  ac0788
  20367616_1  lab-pi-sweep               RUNNING           0:24  ac0788

[tick 3] refreshing in 10 s — interrupt the cell to stop


KeyboardInterrupt: 

## 9. Cancel and inspect

Use `%scancel` to cancel a job and `%scontrol show job` to inspect its full record — including
why it was pending, which node ran it, and when it started and ended.

After a job finishes, `scontrol show job` is only available for a short window (configured by the admin).
For longer history, use `sacct`.

In [26]:
%slurm display raw

# Uncomment and replace <jobid> with an actual job ID:
# %scontrol show job 20367616_2
# %scancel 20367616_2

# Show recent finished jobs (last 1 hour):
%sacct -u $USER --starttime=now-1hour --format=JobID,JobName,State,Elapsed,ExitCode

JobID           JobName      State    Elapsed ExitCode 
------------ ---------- ---------- ---------- -------- 
20367603      lab-hello  COMPLETED   00:00:00      0:0 
20367603.ba+      batch  COMPLETED   00:00:00      0:0 
20367605     lab-dep-g+  COMPLETED   00:00:00      0:0 
20367605.ba+      batch  COMPLETED   00:00:00      0:0 
20367606     lab-dep-s+  COMPLETED   00:00:00      0:0 
20367606.ba+      batch  COMPLETED   00:00:00      0:0 
20367604_1   lab-pi-sw+  COMPLETED   00:00:27      0:0 
20367604_1.+      batch  COMPLETED   00:00:27      0:0 
20367604_2   lab-pi-sw+  COMPLETED   00:00:54      0:0 
20367604_2.+      batch  COMPLETED   00:00:54      0:0 
20367604_3   lab-pi-sw+  COMPLETED   00:01:21      0:0 
20367604_3.+      batch  COMPLETED   00:01:21      0:0 
20367609      lab-hello  COMPLETED   00:00:01      0:0 
20367609.ba+      batch  COMPLETED   00:00:01      0:0 
20367610         lab-pi  COMPLETED   00:00:02      0:0 
20367610.ba+      batch  COMPLETED   00:00:02   

## 10. Takeaways

- `%%sbatch` turns a notebook cell into a batch script — the most natural way to submit from Jupyter.
- Always use `--output` and `--error` to redirect job output to named files you can read back.
- `--array` is the right tool for parameter sweeps — one submission, many tasks, no loop in the notebook.
- `--dependency=afterok:JOBID` chains jobs into pipelines — submit all jobs at once so the dependency is set before A finishes.
- Parsing `squeue` with `subprocess` lets you build status dashboards inside the notebook.
- `plgrid-testing` (MaxSubmitPU=1) is only for single quick jobs; use `plgrid` for arrays and pipelines.